# 03 - Pipeline TensorFlow

Este notebook monta e testa o pipeline de carregamento das imagens com TensorFlow, considerando a divisão atual do projeto em **treino, validação e teste**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

# Caminho raiz do projeto.
# Como este notebook fica dentro da pasta notebooks/, Path("..") aponta para a raiz do projeto.
PROJECT_DIR = Path("..").resolve()
SPLITS_DIR = PROJECT_DIR / "data" / "splits"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_DIR)
print("TensorFlow:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))

## 1. Carregamento dos arquivos de divisão

Os arquivos `train_split.csv`, `val_split.csv` e `test_split.csv` devem ter sido gerados previamente pelo script `src/prepare_splits.py`.

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_split.csv")

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

train_df.head()

In [ ]:
print("Distribuição no treino:")
print(train_df["diagnosis"].value_counts().sort_index())

print("
Distribuição na validação:")
print(val_df["diagnosis"].value_counts().sort_index())

print("
Distribuição no teste:")
print(test_df["diagnosis"].value_counts().sort_index())

## 2. Configurações do pipeline

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

class_names = {
    0: "Sem retinopatia",
    1: "Retinopatia leve",
    2: "Retinopatia moderada",
    3: "Retinopatia severa",
    4: "Retinopatia proliferativa",
}

## 3. Função de carregamento e pré-processamento

A função abaixo lê o arquivo de imagem, decodifica em RGB, redimensiona para `224 x 224`, converte os pixels para `float32`, normaliza para o intervalo `[0, 1]` e transforma o rótulo em one-hot encoding.

In [ ]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    label = tf.cast(label, tf.int32)
    label = tf.one_hot(label, NUM_CLASSES)

    return image, label

## 4. Criação dos datasets TensorFlow

O conjunto de treino pode ser embaralhado. Validação e teste não devem ser embaralhados para facilitar a reprodutibilidade da avaliação.

In [ ]:
def make_dataset(df, shuffle=False):
    image_paths = df["image_path"].astype(str).values
    labels = df["diagnosis"].astype(int).values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

In [ ]:
for images, labels in train_ds.take(1):
    print("Treino - imagens:", images.shape)
    print("Treino - rótulos:", labels.shape)

for images, labels in val_ds.take(1):
    print("Validação - imagens:", images.shape)
    print("Validação - rótulos:", labels.shape)

for images, labels in test_ds.take(1):
    print("Teste - imagens:", images.shape)
    print("Teste - rótulos:", labels.shape)

## 5. Visualização de amostras do treino

Esta etapa confirma visualmente se as imagens estão sendo carregadas com seus rótulos corretos.

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(9):
        label_index = int(np.argmax(labels[i].numpy()))

        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(f"{label_index} - {class_names[label_index]}")
        plt.axis("off")

plt.tight_layout()
plt.show()

## 6. Aumento de dados por transformações geométricas

O aumento de dados é aplicado somente ao conjunto de treinamento. Validação e teste permanecem sem augmentation.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomZoom(0.05, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomTranslation(0.03, 0.03, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomFlip("horizontal"),
], name="data_augmentation")

def apply_augmentation(image, label):
    image = data_augmentation(image, training=True)
    return image, label

train_ds_augmented = train_ds.map(apply_augmentation, num_parallel_calls=AUTOTUNE)

In [ ]:
for images, labels in train_ds_augmented.take(1):
    print("Treino com augmentation - imagens:", images.shape)
    print("Treino com augmentation - rótulos:", labels.shape)

## 7. Figura de exemplos do aumento de dados

A figura abaixo mostra exemplos de transformações aplicadas em imagens de diferentes classes. Ela pode ser usada na metodologia do TCC.

In [ ]:
plt.figure(figsize=(12, 10))
plot_index = 1

for class_id in range(NUM_CLASSES):
    sample_row = train_df[train_df["diagnosis"] == class_id].sample(1, random_state=SEED).iloc[0]

    image_path = sample_row["image_path"]
    label = int(sample_row["diagnosis"])

    image, label_one_hot = load_image(image_path, label)

    for _ in range(3):
        augmented_image = data_augmentation(tf.expand_dims(image, axis=0), training=True)

        plt.subplot(NUM_CLASSES, 3, plot_index)
        plt.imshow(augmented_image[0].numpy())
        plt.title(f"{label} - {class_names[label]}", fontsize=9)
        plt.axis("off")

        plot_index += 1

plt.tight_layout()

fig_path = FIGURES_DIR / "data_augmentation_examples_classes.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figura salva em:", fig_path)

## 8. Checagem final

Se esta célula rodar sem erro, o pipeline está pronto para ser usado nos notebooks de treinamento.

In [ ]:
print("Pipeline TensorFlow concluído com sucesso.")
print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)
print("Imagem:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)